# 💧 LLM-Based Automated Water Pollution Complaint Classification and Routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NaveenAnand01/water-pollution-complaint-llm/blob/main/LLM_Water_Pollution_Complaint_Routing.ipynb)

Citizens report water pollution in free text through portals, apps, helplines and social media. Officers then have to
read each one, decide what kind of problem it is, how urgent it is, and which department must act. This notebook
automates that pipeline:

| Stage | What happens |
|---|---|
| **1. Ingest** | Load the *AquaGrievance-TN* complaint dataset (2,500 labelled complaints, 12 districts of Tamil Nadu) |
| **2. Classify (LLM)** | An instruction-tuned LLM (**Qwen2.5-Instruct**, open-source, runs free on Colab) reads the complaint and picks one of 8 pollution categories via few-shot prompting; class probabilities come from the model's next-token logits |
| **3. Prioritise** | A severity model (TF-IDF + Logistic Regression) plus safety-keyword escalation assigns **High / Medium / Low** priority and an SLA |
| **4. Route** | Category → responsible department; low-confidence predictions are flagged for human review |
| **5. Evaluate** | LLM vs. classical TF-IDF baseline on a test set written with **unseen phrasings** |
| **6. Dashboard** | Interactive Plotly charts + a Gradio web dashboard with a live complaint classifier |

> **Runtime:** `Runtime → Change runtime type → T4 GPU` is recommended (≈5 min end-to-end). It also works on CPU with a smaller model and evaluation sample.

## 1. Setup

In [ ]:
!pip -q install "gradio>=6,<7" plotly scikit-learn pandas "transformers>=4.44" accelerate

In [ ]:
import os, re, json, time, warnings, textwrap
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

GITHUB_REPO = "NaveenAnand01/water-pollution-complaint-llm"
DATA_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/data/water_pollution_complaints.csv"

# ---- palette (validated categorical order + fixed status colours) ----
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
PRIORITY_COLORS = {"High": "#d03b3b", "Medium": "#fab219", "Low": "#0ca30c"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
FONT = "Inter, Segoe UI, Roboto, sans-serif"

def style(fig, h=380, legend=True):
    fig.update_layout(template="plotly_white", height=h, font=dict(family=FONT, color=INK, size=13),
                      margin=dict(l=10, r=10, t=50, b=10), title_font=dict(size=16),
                      plot_bgcolor="#fcfcfb", paper_bgcolor="#ffffff", showlegend=legend,
                      hoverlabel=dict(font_family=FONT, bgcolor="white"),
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    return fig
print("✅ libraries ready")

## 2. Dataset — *AquaGrievance-TN: Water Pollution Complaint Dataset*

2,500 citizen complaints (Jan 2025 – Oct 2026) across 12 Tamil Nadu districts and 40+ rivers, lakes and canals.
Each row has the complaint text and gold labels for **category**, **responsible department**, **priority** and **SLA**.
The dataset is generated reproducibly by `data/generate_dataset.py` in the repo, with realistic noise (typos, lowercase,
Tanglish fillers, distracting side-mentions of other problems). The **test split uses sentence templates never seen
in training**, so it measures real generalisation rather than memorisation.

In [ ]:
try:
    df = pd.read_csv(DATA_URL)
    print("Loaded from GitHub:", DATA_URL)
except Exception as e:
    df = pd.read_csv("data/water_pollution_complaints.csv")   # local fallback
    print("Loaded local copy")
df["reported_at"] = pd.to_datetime(df["reported_at"])
print(df.shape)
df.head(8)

In [ ]:
CATEGORIES = list(df.drop_duplicates("category").sort_values("category").category)
DEPT = dict(df.drop_duplicates("category")[["category", "department"]].values)
CAT_COLOR = {c: SERIES[i] for i, c in enumerate(CATEGORIES)}
SLA_HOURS = {"High": 24, "Medium": 72, "Low": 168}
train_df, test_df = df[df.split == "train"].reset_index(drop=True), df[df.split == "test"].reset_index(drop=True)
print(f"train={len(train_df)}  test={len(test_df)}")
pd.DataFrame({"category": CATEGORIES, "routed_to": [DEPT[c] for c in CATEGORIES],
              "complaints": [int((df.category == c).sum()) for c in CATEGORIES]})

## 3. Exploratory analysis

In [ ]:
cc = df.category.value_counts().sort_values()
fig = go.Figure(go.Bar(x=cc.values, y=cc.index, orientation="h", marker_color=[CAT_COLOR[c] for c in cc.index],
                       text=cc.values, textposition="outside", hovertemplate="%{y}: %{x} complaints<extra></extra>"))
style(fig, 380, False).update_layout(title="Complaints by pollution category").show()

pc = df.priority.value_counts().reindex(["High", "Medium", "Low"])
fig = go.Figure(go.Pie(labels=pc.index, values=pc.values, hole=.6, sort=False,
                       marker=dict(colors=[PRIORITY_COLORS[p] for p in pc.index], line=dict(color="white", width=2)),
                       textinfo="label+percent"))
style(fig, 340).update_layout(title="Priority mix").show()

m = df.set_index("reported_at").groupby([pd.Grouper(freq="MS"), "priority"]).size().reset_index(name="n")
fig = px.line(m, x="reported_at", y="n", color="priority", color_discrete_map=PRIORITY_COLORS, markers=True)
style(fig, 360).update_traces(line_width=2).update_layout(title="Monthly complaint volume by priority", xaxis_title="", yaxis_title="complaints").show()
df["words"] = df.complaint_text.str.split().str.len()
print("Average complaint length:", round(df.words.mean(), 1), "words")

## 4. LLM classifier (few-shot, open-source Qwen2.5-Instruct)

* The prompt lists the 8 categories with short definitions and tells the model to label the **root cause** (ignoring side-mentions).
* One short example per category from the *train* split is given as few-shot chat turns.
* Instead of free-form generation, we read the logits of the answer tokens **"1"…"8"** → a probability for each category in one forward pass. This is fast, never produces an invalid label, and gives a **confidence score** used for routing.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = os.environ.get("MODEL_ID") or ("Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct")
N_EVAL = None if DEVICE == "cuda" else 150      # CPU: evaluate on a sample to keep it quick
BATCH = 16 if DEVICE == "cuda" else 4

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
DIGIT_IDS = [tok.encode(str(i + 1), add_special_tokens=False)[0] for i in range(len(CATEGORIES))]
print(f"Loaded {MODEL_ID} on {DEVICE.upper()} in {time.time()-t0:.0f}s")

In [ ]:
DEFINITIONS = {
    "Agricultural Runoff": "pesticides, fertilisers, farm/poultry/cattle waste washing from fields into water",
    "Algal Bloom & Fish Kill": "green algae, water hyacinth, scum, mass death of fish in a water body",
    "Drinking Water Contamination": "problem with SUPPLIED tap / pipeline / overhead-tank / RO drinking water",
    "Groundwater Contamination": "borewell, open well or hand-pump water that is polluted, salty, high fluoride/nitrate/iron",
    "Industrial Effluent Discharge": "factories, dyeing units, tanneries, mills releasing effluent or process water",
    "Oil & Chemical Spill": "oil, diesel, petrol, acid or chemical leaking/spilled from tankers, pipelines, drums, boats",
    "Sewage & Drainage Overflow": "sewage, manholes, underground drainage, septic waste, domestic waste water",
    "Solid Waste & Plastic Dumping": "garbage, plastic, debris, market/meat/medical waste dumped in or near water",
}
SYSTEM = ("You are an environmental grievance officer who classifies citizen complaints about water pollution.\n"
          "Categories:\n" + "\n".join(f"{i+1}. {c} - {DEFINITIONS[c]}" for i, c in enumerate(CATEGORIES)) +
          "\nLabel the ROOT CAUSE of the complaint; ignore minor side-mentions of other problems. "
          "Reply with only the category number.")

# one short few-shot example per category (from train split only)
shots = (train_df[train_df.complaint_text.str.len() < 170]
         .groupby("category").sample(1, random_state=7))
FEWSHOT = []
for _, r in shots.sample(frac=1, random_state=3).iterrows():
    FEWSHOT += [{"role": "user", "content": f"Complaint: {r.complaint_text}"},
                {"role": "assistant", "content": str(CATEGORIES.index(r.category) + 1)}]

def build_prompt(text):
    msgs = [{"role": "system", "content": SYSTEM}] + FEWSHOT + [{"role": "user", "content": f"Complaint: {text}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def llm_probs(texts, batch=BATCH):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok([build_prompt(t) for t in texts[i:i+batch]], return_tensors="pt", padding=True).to(DEVICE)
        logits = llm(**enc).logits[:, -1, :]
        out.append(torch.softmax(logits[:, DIGIT_IDS].float(), dim=-1).cpu().numpy())
    return np.vstack(out)

p = llm_probs(["Dead fish everywhere in the lake and the water is green"])[0]
print("Sanity check →", CATEGORIES[p.argmax()], f"({p.max():.0%})")

## 5. Priority model + routing engine

In [ ]:
# severity model: TF-IDF + Logistic Regression trained on the train split
prio_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
prio_clf = LogisticRegression(max_iter=3000, C=5, class_weight="balanced")
prio_clf.fit(prio_vec.fit_transform(train_df.complaint_text), train_df.priority)

ESCALATE = re.compile(r"hospital|died|dead birds|children|vomit|diarrh|jaundice|emergency|breathing|danger|fumes|burning eyes", re.I)

def predict_priority(texts, cats):
    pr = prio_clf.predict(prio_vec.transform(texts))
    final = []
    for t, c, p in zip(texts, cats, pr):
        if ESCALATE.search(t):
            p = "High"                                   # safety override
        elif c == "Oil & Chemical Spill" and p == "Low":
            p = "Medium"                                 # hazmat never low
        final.append(p)
    return np.array(final)

REVIEW_THRESHOLD = 0.55

def route(texts, probs):
    cats = [CATEGORIES[i] for i in probs.argmax(1)]
    pri = predict_priority(texts, cats)
    conf = probs.max(1)
    return pd.DataFrame({
        "predicted_category": cats, "confidence": conf.round(3), "routed_to": [DEPT[c] for c in cats],
        "predicted_priority": pri, "assigned_sla_h": [SLA_HOURS[p] for p in pri],
        "status": np.where(conf < REVIEW_THRESHOLD, "⚠️ Human review", "✅ Auto-routed")})
print("routing engine ready")

## 6. Run the pipeline on the held-out test set

In [ ]:
eval_df = test_df if N_EVAL is None else test_df.sample(N_EVAL, random_state=0).reset_index(drop=True)
t0 = time.time()
P = llm_probs(eval_df.complaint_text.tolist())
print(f"LLM classified {len(eval_df)} complaints in {time.time()-t0:.1f}s")
routed = pd.concat([eval_df.reset_index(drop=True), route(eval_df.complaint_text.tolist(), P)], axis=1)
routed.insert(0, "ticket_id", ["TKT-" + c.split("-")[1] for c in routed.complaint_id])
routed["due_by"] = routed.reported_at + pd.to_timedelta(routed.assigned_sla_h, unit="h")
routed[["ticket_id", "complaint_text", "predicted_category", "confidence", "routed_to", "predicted_priority", "status"]].head(10)

## 7. Evaluation — LLM vs. classical baseline

In [ ]:
# Baseline: TF-IDF + Logistic Regression for category (same train split)
base_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
base_clf = LogisticRegression(max_iter=3000, C=5).fit(base_vec.fit_transform(train_df.complaint_text), train_df.category)
base_pred = base_clf.predict(base_vec.transform(eval_df.complaint_text))

y, llm_pred = routed.category, routed.predicted_category
metrics = pd.DataFrame({
    "Model": ["TF-IDF + LogReg (baseline)", f"LLM few-shot ({MODEL_ID.split('/')[1]})"],
    "Accuracy": [accuracy_score(y, base_pred), accuracy_score(y, llm_pred)],
    "Macro F1": [f1_score(y, base_pred, average="macro"), f1_score(y, llm_pred, average="macro")],
}).round(3)
prio_acc = accuracy_score(routed.priority, routed.predicted_priority)
route_acc = accuracy_score(routed.department, routed.routed_to)
auto = (routed.status.str.contains("Auto")).mean()
_a = routed[routed.status.str.contains("Auto")]
auto_acc = accuracy_score(_a.category, _a.predicted_category) if len(_a) else float("nan")
print(metrics.to_string(index=False))
print(f"\nRouting accuracy (correct department): {route_acc:.1%}")
print(f"Priority accuracy: {prio_acc:.1%}")
print(f"Auto-routed: {auto:.1%} of complaints, with {auto_acc:.1%} accuracy; the rest go to human review")
print("\n", classification_report(y, llm_pred, digits=3))

In [ ]:
cm = confusion_matrix(y, llm_pred, labels=CATEGORIES)
short = [c.split(" ")[0] if len(c) > 18 else c for c in CATEGORIES]
fig = go.Figure(go.Heatmap(z=cm, x=short, y=short, colorscale=[[0, "#f4f8fd"], [1, "#1c4f8f"]], text=cm,
                           texttemplate="%{text}", hovertemplate="true %{y} → predicted %{x}: %{z}<extra></extra>", showscale=False))
style(fig, 480, False).update_layout(title="LLM confusion matrix (rows = true, columns = predicted)",
                                     yaxis_autorange="reversed").show()

## 8. 📊 Dashboard
Run the next cell to open the interactive dashboard (inline in Colab, plus a public `gradio.live` link you can share).

In [ ]:
import gradio as gr

def kpi(label, value, sub="", color="#2a78d6"):
    return f"""<div class='kpi' style='border-top:4px solid {color}'><div class='kpi-l'>{label}</div>
               <div class='kpi-v'>{value}</div><div class='kpi-s'>{sub}</div></div>"""

def kpi_row():
    return "<div class='kpis'>" + "".join([
        kpi("Total complaints", f"{len(df):,}", "Jan 2025 – Oct 2026"),
        kpi("High priority", f"{(df.priority=='High').mean():.0%}", f"{(df.priority=='High').sum():,} need action in 24 h", PRIORITY_COLORS["High"]),
        kpi("LLM accuracy", f"{metrics.Accuracy.iloc[1]:.1%}", f"baseline {metrics.Accuracy.iloc[0]:.1%}", "#1baf7a"),
        kpi("Auto-routed", f"{auto:.0%}", f"{auto_acc:.1%} accurate", "#6250d6"),
        kpi("Departments", f"{len(DEPT)}", "routing targets", "#eb6834"),
    ]) + "</div>"

# ---------- overview figures ----------
def fig_category():
    cc = df.category.value_counts().sort_values()
    f = go.Figure(go.Bar(x=cc.values, y=cc.index, orientation="h", marker_color=[CAT_COLOR[c] for c in cc.index],
                         hovertemplate="%{y}: %{x}<extra></extra>"))
    return style(f, 360, False).update_layout(title="Complaints by category")

def fig_district():
    d = df.groupby(["district", "priority"]).size().reset_index(name="n")
    order = df.district.value_counts().index.tolist()
    f = px.bar(d, x="district", y="n", color="priority", color_discrete_map=PRIORITY_COLORS,
               category_orders={"district": order, "priority": ["High", "Medium", "Low"]})
    return style(f, 360).update_layout(title="Complaints by district", xaxis_title="", yaxis_title="", bargap=.25)

def fig_trend():
    return style(px.line(m, x="reported_at", y="n", color="priority", color_discrete_map=PRIORITY_COLORS), 330)\
        .update_layout(title="Monthly trend", xaxis_title="", yaxis_title="").update_traces(line_width=2)

def fig_channel():
    c = df.channel.value_counts()
    f = go.Figure(go.Pie(labels=c.index, values=c.values, hole=.6, marker=dict(colors=SERIES[:len(c)], line=dict(color="white", width=2))))
    return style(f, 330).update_layout(title="Reporting channel")

def fig_waterbodies():
    w = df.water_body.value_counts().head(10).sort_values()
    f = go.Figure(go.Bar(x=w.values, y=w.index, orientation="h", marker_color="#2a78d6", hovertemplate="%{y}: %{x}<extra></extra>"))
    return style(f, 360, False).update_layout(title="Top 10 most-reported water bodies")

# ---------- model figures ----------
def fig_compare():
    mm = metrics.melt(id_vars="Model", var_name="Metric", value_name="Score")
    f = px.bar(mm, x="Metric", y="Score", color="Model", barmode="group", text_auto=".1%",
               color_discrete_sequence=["#9aa7b8", "#1baf7a"])
    return style(f, 360).update_layout(title="LLM vs baseline (unseen-phrasing test set)", yaxis_tickformat=".0%", yaxis_range=[0, 1.05])

def fig_cm():
    return style(go.Figure(go.Heatmap(z=cm, x=short, y=short, colorscale=[[0, "#f4f8fd"], [1, "#1c4f8f"]], text=cm,
                 texttemplate="%{text}", showscale=False)), 420, False).update_layout(title="Confusion matrix", yaxis_autorange="reversed")

def fig_f1():
    rep = classification_report(y, llm_pred, output_dict=True)
    s = pd.Series({c: rep[c]["f1-score"] for c in CATEGORIES}).sort_values()
    f = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=[CAT_COLOR[c] for c in s.index],
                         text=[f"{v:.2f}" for v in s.values], textposition="outside"))
    return style(f, 420, False).update_layout(title="Per-category F1 (LLM)", xaxis_range=[0, 1.1])

def fig_conf_hist():
    f = px.histogram(routed, x="confidence", nbins=25, color_discrete_sequence=["#6250d6"])
    f.add_vline(x=REVIEW_THRESHOLD, line_dash="dash", line_color="#d03b3b", annotation_text="review threshold")
    return style(f, 330, False).update_layout(title="LLM confidence distribution", yaxis_title="complaints")

# ---------- routing ----------
def fig_dept_load(frame):
    d = frame.groupby(["routed_to", "predicted_priority"]).size().reset_index(name="n")
    d["dept"] = d.routed_to.str.split(" - ").str[0]
    f = px.bar(d, y="dept", x="n", color="predicted_priority", orientation="h", color_discrete_map=PRIORITY_COLORS,
               category_orders={"predicted_priority": ["High", "Medium", "Low"]})
    return style(f, 380).update_layout(title="Department workload (routed tickets)", yaxis_title="", xaxis_title="tickets", bargap=.3)

QUEUE_COLS = ["ticket_id", "reported_at", "district", "complaint_text", "predicted_category", "confidence",
              "predicted_priority", "routed_to", "due_by", "status"]

def filter_queue(dept, prio, status):
    f = routed.copy()
    if dept != "All": f = f[f.routed_to == dept]
    if prio != "All": f = f[f.predicted_priority == prio]
    if status != "All": f = f[f.status == status]
    f = f.sort_values(["predicted_priority", "due_by"], key=lambda s: s.map({"High": 0, "Medium": 1, "Low": 2}) if s.name == "predicted_priority" else s)
    out = f[QUEUE_COLS].copy()
    out["reported_at"] = out.reported_at.dt.strftime("%d %b %Y %H:%M"); out["due_by"] = out.due_by.dt.strftime("%d %b %Y %H:%M")
    return out, fig_dept_load(f if len(f) else routed), f"**{len(f)}** tickets match"

# ---------- live classifier ----------
@torch.no_grad()
def explain(text, cat):
    msgs = [{"role": "system", "content": "You are an environmental grievance officer. Answer in one short sentence."},
            {"role": "user", "content": f"Complaint: {text}\nIt was classified as '{cat}'. In one sentence, explain why and what action the department should take."}]
    enc = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(DEVICE)
    out = llm.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

def classify_live(text, district):
    if not text or len(text.strip()) < 10:
        return "<p>Please type a complaint (at least a sentence).</p>", None
    probs = llm_probs([text])[0]
    r = route([text], probs[None, :]).iloc[0]
    reason = explain(text, r.predicted_category)
    pc = PRIORITY_COLORS[r.predicted_priority]
    tid = f"TKT-LIVE-{int(time.time()) % 100000:05d}"
    card = f"""<div class='ticket'>
      <div class='t-head'><span>🎫 {tid}</span><span class='badge' style='background:{pc}'>{r.predicted_priority} priority</span></div>
      <div class='t-grid'>
        <div><div class='t-l'>Category</div><div class='t-v'>{r.predicted_category}</div></div>
        <div><div class='t-l'>Routed to</div><div class='t-v'>{r.routed_to}</div></div>
        <div><div class='t-l'>District</div><div class='t-v'>{district}</div></div>
        <div><div class='t-l'>Resolve within</div><div class='t-v'>{r.assigned_sla_h} hours</div></div>
        <div><div class='t-l'>LLM confidence</div><div class='t-v'>{r.confidence:.0%}</div></div>
        <div><div class='t-l'>Status</div><div class='t-v'>{r.status}</div></div>
      </div>
      <div class='t-reason'>🤖 <b>LLM reasoning:</b> {reason}</div></div>"""
    s = pd.Series(probs, index=CATEGORIES).sort_values()
    f = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=[CAT_COLOR[c] for c in s.index],
                         text=[f"{v:.0%}" for v in s.values], textposition="outside"))
    return card, style(f, 340, False).update_layout(title="Category probabilities", xaxis_tickformat=".0%", xaxis_range=[0, 1.12])

EXAMPLES = [
    ["Blue coloured foamy water is coming from the dyeing unit outlet straight into the Noyyal river near Tiruppur, cattle drank it and died", "Tiruppur"],
    ["sir our corporation tap water is coming muddy and smelly for one week, my kids got fever and loose motion", "Tiruchirappalli"],
    ["A diesel tanker toppled near the bridge and oil is floating on the Cauvery, strong smell", "Erode"],
    ["People throw plastic covers and hotel waste into the temple tank every morning, kindly clean it", "Madurai"],
    ["Lot of dead fish on the lake shore today and the water has turned dark green", "Coimbatore"],
]

CSS = """
.gradio-container {max-width: 1280px !important; margin: auto; font-family: Inter, Segoe UI, Roboto, sans-serif;}
#hero {background: linear-gradient(120deg, #0b3d6b 0%, #1170a8 55%, #1baf9a 100%); color: #fff; padding: 26px 30px;
       border-radius: 16px; margin-bottom: 6px;}
#hero h1 {margin: 0; font-size: 28px; color: #fff;} #hero p {margin: 6px 0 0; opacity: .92; font-size: 15px; color: #fff;}
.kpis {display: grid; grid-template-columns: repeat(5, 1fr); gap: 14px; margin: 6px 0 4px;}
.kpi {background: #fff; border-radius: 12px; padding: 14px 16px; box-shadow: 0 1px 3px rgba(0,0,0,.08);}
.kpi-l {font-size: 12px; color: #52514e; text-transform: uppercase; letter-spacing: .04em;}
.kpi-v {font-size: 28px; font-weight: 700; color: #0b0b0b; margin-top: 4px;} .kpi-s {font-size: 12px; color: #52514e;}
.ticket {border: 1px solid #dfe6ee; border-radius: 14px; padding: 18px; background: #fff; box-shadow: 0 2px 8px rgba(11,61,107,.08);}
.t-head {display: flex; justify-content: space-between; align-items: center; font-weight: 700; font-size: 17px; color:#0b0b0b; margin-bottom: 12px;}
.badge {color: #fff; padding: 4px 12px; border-radius: 999px; font-size: 13px;}
.t-grid {display: grid; grid-template-columns: 1fr 1fr; gap: 12px 18px;}
.t-l {font-size: 12px; color: #52514e; text-transform: uppercase;} .t-v {font-size: 15px; font-weight: 600; color: #0b0b0b;}
.t-reason {margin-top: 14px; padding: 12px; background: #f1f7fb; border-radius: 10px; font-size: 14px; color: #0b0b0b;}
@media (max-width: 900px) {.kpis {grid-template-columns: repeat(2, 1fr);}}
"""

with gr.Blocks(title="AquaRoute – Water Pollution Complaint Dashboard") as demo:
    gr.HTML(f"""<div id='hero'><h1>💧 AquaRoute: LLM Water Pollution Complaint Classification & Routing</h1>
               <p>{len(df):,} citizen complaints · {df.district.nunique()} districts · {len(CATEGORIES)} pollution categories ·
               model: {MODEL_ID}</p></div>""")
    gr.HTML(kpi_row())
    with gr.Tabs():
        with gr.Tab("📝 Live Classifier"):
            with gr.Row():
                with gr.Column(scale=5):
                    inp = gr.Textbox(label="Citizen complaint", lines=5, placeholder="Describe the water pollution problem…")
                    dist = gr.Dropdown(sorted(df.district.unique()), value="Tiruchirappalli", label="District")
                    btn = gr.Button("Classify & route", variant="primary")
                    gr.Examples(EXAMPLES, [inp, dist])
                with gr.Column(scale=6):
                    card = gr.HTML("<div class='ticket'>The routed ticket will appear here.</div>")
                    probs_plot = gr.Plot()
            btn.click(classify_live, [inp, dist], [card, probs_plot])
        with gr.Tab("📊 Overview"):
            with gr.Row():
                gr.Plot(fig_category()); gr.Plot(fig_district())
            with gr.Row():
                gr.Plot(fig_trend()); gr.Plot(fig_channel())
            gr.Plot(fig_waterbodies())
        with gr.Tab("🚦 Routing Queue"):
            with gr.Row():
                d_dd = gr.Dropdown(["All"] + sorted(DEPT.values()), value="All", label="Department")
                p_dd = gr.Dropdown(["All", "High", "Medium", "Low"], value="All", label="Priority")
                s_dd = gr.Dropdown(["All", "✅ Auto-routed", "⚠️ Human review"], value="All", label="Status")
            cnt = gr.Markdown()
            load_plot = gr.Plot()
            table = gr.Dataframe(wrap=True, max_height=480)
            for c in (d_dd, p_dd, s_dd):
                c.change(filter_queue, [d_dd, p_dd, s_dd], [table, load_plot, cnt])
            demo.load(filter_queue, [d_dd, p_dd, s_dd], [table, load_plot, cnt])
        with gr.Tab("🤖 Model Performance"):
            gr.Markdown(f"Evaluated on **{len(eval_df)} held-out complaints written with unseen phrasings**. "
                        f"Routing accuracy **{route_acc:.1%}**, priority accuracy **{prio_acc:.1%}**, "
                        f"auto-routed **{auto:.0%}** at **{auto_acc:.1%}** accuracy.")
            with gr.Row():
                gr.Plot(fig_compare()); gr.Plot(fig_conf_hist())
            with gr.Row():
                gr.Plot(fig_cm()); gr.Plot(fig_f1())
            gr.Dataframe(metrics)

demo.launch(share=True, debug=False, theme=gr.themes.Soft(primary_hue="cyan", secondary_hue="blue"), css=CSS)

## 9. Export results
Saves the routed ticket queue and the metrics so they can be downloaded or committed back to the repo.

In [ ]:
os.makedirs("outputs", exist_ok=True)
routed.to_csv("outputs/routed_tickets.csv", index=False)
metrics.to_csv("outputs/model_metrics.csv", index=False)
print("Saved outputs/routed_tickets.csv and outputs/model_metrics.csv")

## 10. Conclusion
* A small open-source LLM, prompted few-shot and read through its answer-token probabilities, classifies free-text water-pollution complaints into 8 categories and **generalises to unseen phrasings better than a TF-IDF baseline**.
* Combining the LLM with a severity model and safety-keyword escalation gives each ticket a department, priority and SLA automatically; **low-confidence cases are sent to a human**, keeping the system safe.
* The dashboard gives officers a live classifier, an SLA-sorted routing queue, and monitoring of model quality.

**Future work:** multilingual (Tamil) complaints, image evidence with a vision-language model, geo-mapping of hotspots, and fine-tuning (LoRA) on real grievance-portal data.